
1. Load movies.csv as movies dataframe and ratings.csv as ratings dataframe. Use info function to view the schema, null counts, and memory usage of each data frames
2. How many records are there in movies and ratings dataframe [movies: 9125, ratings: 100004]
3. Find the movies, for which title contains Godfather? [Hint: use movies.title.str.contains() to find substring in title]
4. How many ratings the movie "Godfather, The (1972)" has received?[Answer: 200]
5. Using ratings dataframe, calculate average rating and number of ratings for each movie. What is the average rating for "Godfather, The (1972)" [Answer: 4.487]
6. Select all movie Ids that have received more than 100 ratings. How many movies are there having more than 100 ratings? [Answer: 149]
7. From the output of step 5 and 6, take top 10 records with highest avg rating. [Hint: use sort and iloc] 
8. Merge the output of 7 with movies dataframe to show title. [Hint: call pd.merge function]
9. For each genre find top 3 movies. Genres column contain multiple values separated by pipe ('|'). Use explode (df.explode(...) function to generate one for each genre repeating the other values in the same row.

https://raw.githubusercontent.com/abulbasar/data/master/movielens/movies.csv
https://raw.githubusercontent.com/abulbasar/data/master/movielens/ratings.csv


In [1]:
import pandas as pd

In [2]:
movies = pd.read_csv("https://raw.githubusercontent.com/abulbasar/data/master/movielens/movies.csv")
ratings = pd.read_csv("https://raw.githubusercontent.com/abulbasar/data/master/movielens/ratings.csv")

In [3]:
movies.shape

(9125, 3)

In [4]:
len(movies)

9125

In [5]:
ratings.shape

(100004, 4)

In [7]:
movies[movies.title.str.contains('Godfather')]

,movieId,title,genres
695,858,"Godfather, The (1972)",Crime|Drama
977,1221,"Godfather: Part II, The (1974)",Crime|Drama
1585,2023,"Godfather: Part III, The (1990)",Crime|Drama|Mystery|Thriller
5470,8607,Tokyo Godfathers (2003),Adventure|Animation|Drama


In [8]:
movies[movies.title.str.lower().str.contains('godfather')]

,movieId,title,genres
695,858,"Godfather, The (1972)",Crime|Drama
977,1221,"Godfather: Part II, The (1974)",Crime|Drama
1585,2023,"Godfather: Part III, The (1990)",Crime|Drama|Mystery|Thriller
5470,8607,Tokyo Godfathers (2003),Adventure|Animation|Drama


In [10]:
movies[movies.title.apply(lambda r: 'godfather' in r.lower())]

,movieId,title,genres
695,858,"Godfather, The (1972)",Crime|Drama
977,1221,"Godfather: Part II, The (1974)",Crime|Drama
1585,2023,"Godfather: Part III, The (1990)",Crime|Drama|Mystery|Thriller
5470,8607,Tokyo Godfathers (2003),Adventure|Animation|Drama


In [11]:
movies[movies.title == 'Godfather, The (1972)']

,movieId,title,genres
695,858,"Godfather, The (1972)",Crime|Drama


In [14]:
movie_id = movies[movies.title == 'Godfather, The (1972)'].movieId.values[0]
movie_id

np.int64(858)

In [15]:
ratings[ratings.movieId == movie_id]

,userId,movieId,rating,timestamp
180,4,858,5.0,949779022
370,5,858,2.5,1163373651
602,8,858,5.0,1154400181
1140,15,858,5.0,997938703
2730,17,858,5.0,1127469000
...,...,...,...,...
96768,648,858,3.5,1138477822
97574,654,858,4.5,1145391858
98186,656,858,5.0,986241026
98522,659,858,5.0,860395342


In [17]:
movie_avg = ratings.groupby("movieId").rating.agg(["mean", "count"])
movie_avg

,mean,count
movieId,,
1,3.872470,247
2,3.401869,107
3,3.161017,59
4,2.384615,13
5,3.267857,56
...,...,...
161944,5.000000,1
162376,4.500000,1
162542,5.000000,1


In [18]:
movie_avg.loc[movie_id]

mean       4.4875
count    200.0000
Name: 858, dtype: float64

In [20]:
movie_avg[movie_avg['count']>100]

,mean,count
movieId,,
1,3.872470,247
2,3.401869,107
6,3.884615,104
10,3.450820,122
25,3.742574,101
...,...,...
7438,3.665049,103
8961,3.861111,126
33794,3.857143,105


In [24]:
top10 = movie_avg[movie_avg['count']>100].sort_values('mean', ascending = False).iloc[:10]
top10

,mean,count
movieId,,
858,4.487500,200
318,4.487138,311
1221,4.385185,135
50,4.370647,201
527,4.303279,244
1193,4.256944,144
608,4.256696,224
296,4.256173,324
2858,4.236364,220


In [26]:
top10 = top10.merge(movies, left_index=True, right_on='movieId')
top10

,mean,count,movieId,title,genres
695,4.487500,200,858,"Godfather, The (1972)",Crime|Drama
284,4.487138,311,318,"Shawshank Redemption, The (1994)",Crime|Drama
977,4.385185,135,1221,"Godfather: Part II, The (1974)",Crime|Drama
48,4.370647,201,50,"Usual Suspects, The (1995)",Crime|Mystery|Thriller
472,4.303279,244,527,Schindler's List (1993),Drama|War
951,4.256944,144,1193,One Flew Over the Cuckoo's Nest (1975),Drama
535,4.256696,224,608,Fargo (1996),Comedy|Crime|Drama|Thriller
266,4.256173,324,296,Pulp Fiction (1994),Comedy|Crime|Drama|Thriller
2288,4.236364,220,2858,American Beauty (1999),Drama|Romance
6916,4.235537,121,58559,"Dark Knight, The (2008)",Action|Crime|Drama|IMAX


In [30]:
a = movie_avg[movie_avg['count']>100].merge(movies, left_index=True, right_on='movieId')
a['genres'] = a['genres'].apply(lambda s: s.split('|'))
a

,mean,count,movieId,title,genres
0,3.872470,247,1,Toy Story (1995),"[Adventure, Animation, Children, Comedy, Fantasy]"
1,3.401869,107,2,Jumanji (1995),"[Adventure, Children, Fantasy]"
5,3.884615,104,6,Heat (1995),"[Action, Crime, Thriller]"
9,3.450820,122,10,GoldenEye (1995),"[Action, Adventure, Thriller]"
24,3.742574,101,25,Leaving Las Vegas (1995),"[Drama, Romance]"
...,...,...,...,...,...
5152,3.665049,103,7438,Kill Bill: Vol. 2 (2004),"[Action, Drama, Thriller]"
5626,3.861111,126,8961,"Incredibles, The (2004)","[Action, Adventure, Animation, Children, Comedy]"
6158,3.857143,105,33794,Batman Begins (2005),"[Action, Crime, IMAX]"
6916,4.235537,121,58559,"Dark Knight, The (2008)","[Action, Crime, Drama, IMAX]"


In [34]:
a = pd.DataFrame({'a': [1, 2], 'b': [['Drama', 'Romance'], ['Action', 'Crime', 'IMAX']]})
a

,a,b
0,1,"[Drama, Romance]"
1,2,"[Action, Crime, IMAX]"


In [35]:
a.explode('b')

,a,b
0,1,Drama
0,1,Romance
1,2,Action
1,2,Crime
1,2,IMAX


In [38]:
a = movie_avg[movie_avg['count']>100].merge(movies, left_index=True, right_on='movieId')
a['genres'] = a['genres'].apply(lambda s: s.split('|'))
a = a.explode('genres')
a

,mean,count,movieId,title,genres
0,3.87247,247,1,Toy Story (1995),Adventure
0,3.87247,247,1,Toy Story (1995),Animation
0,3.87247,247,1,Toy Story (1995),Children
0,3.87247,247,1,Toy Story (1995),Comedy
0,3.87247,247,1,Toy Story (1995),Fantasy
...,...,...,...,...,...
7575,4.04955,111,79132,Inception (2010),Drama
7575,4.04955,111,79132,Inception (2010),Mystery
7575,4.04955,111,79132,Inception (2010),Sci-Fi
7575,4.04955,111,79132,Inception (2010),Thriller


In [43]:
a['rank'] = a.groupby('genres')['mean'].rank(method='min', ascending=False)
a.query('rank <= 3').sort_values(['genres', 'mean'], ascending= False)


,mean,count,movieId,title,genres,rank
523,3.717822,202,590,Dances with Wolves (1990),Western,1.0
472,4.303279,244,527,Schindler's List (1993),War,1.0
626,4.209524,105,750,Dr. Strangelove or: How I Learned to Stop Worr...,War,2.0
964,4.138393,112,1208,Apocalypse Now (1979),War,3.0
48,4.370647,201,50,"Usual Suspects, The (1995)",Thriller,1.0
535,4.256696,224,608,Fargo (1996),Thriller,2.0
266,4.256173,324,296,Pulp Fiction (1994),Thriller,3.0
953,4.232906,234,1196,Star Wars: Episode V - The Empire Strikes Back...,Sci-Fi,1.0
232,4.221649,291,260,Star Wars: Episode IV - A New Hope (1977),Sci-Fi,2.0
2062,4.183398,259,2571,"Matrix, The (1999)",Sci-Fi,3.0
